## Runtime Compatibility Check

Run this notebook from the repo root or from a Kaggle working directory that contains the repo. The notebook is intentionally thin and delegates training, evaluation, and metrics work to the rebuild-track scripts.


In [ ]:
import os
import platform
import sys
from pathlib import Path


def find_repo_root() -> Path:
    candidates = [
        Path("/kaggle/input/hndsr-mini-project-code/Mini Project"),
        Path("/kaggle/working/HNDSR-Rebuild"),
        Path("/kaggle/working/Mini Project"),
        Path("/kaggle/working"),
        Path.cwd(),
    ]

    print("Searching for repo root...")
    for candidate in candidates:
        exists = candidate.exists()
        has_repo = exists and (candidate / "src").exists() and (candidate / "configs").exists() and (candidate / "scripts").exists()
        print(f"  {candidate}: exists={exists}, has_repo={has_repo}")
        if exists:
            try:
                contents = [child.name for child in list(candidate.iterdir())[:10]]
                print(f"    Contents: {contents}")
            except Exception as exc:
                print(f"    Error listing: {exc}")
        if has_repo:
            return candidate

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return Path.cwd()


REPO_ROOT = find_repo_root()
PYTHON = sys.executable
print(f"\nRepo root: {REPO_ROOT}")
print(f"Python: {PYTHON}")
print(f"Platform: {platform.platform()}")
print(f"Kaggle runtime type: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'local-or-unknown')}")
assert (REPO_ROOT / 'src').exists() and (REPO_ROOT / 'configs').exists(), 'Expected standalone rebuild repo under repo root.'

## Post-Restart GPU Sanity Check

Run the next cell after any runtime restart. If CUDA is not available, continue on CPU only for debugging and document the limitation in the handoff note.


In [ ]:
import torch

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print('Running without CUDA. Keep this run diagnostic-focused if you stay on CPU.')


# vR.1 HNDSR

## Scratch SR3 Kaggle Baseline

This is the first immutable rebuild-track notebook. It is a Kaggle execution shell around repo-owned code, not a notebook-only model implementation.


## Experiment Registry

| Field | Value |
| --- | --- |
| Notebook stem | `vR.1_HNDSR` |
| Lineage | scratch (`vR.x`) |
| Dataset lane | Kaggle `4x-satellite-image-super-resolution` |
| Control lane | Bicubic |
| Trainable lane | SR3 baseline |
| Tracking default | W&B offline |
| Promotion gate | Successful Kaggle run plus review doc |


## Paper Lineage and Hypothesis

- Near-term goal: prove the scratch SR3 pipeline can run reproducibly on Kaggle using repo scripts.
- Medium-term goal: use this stable Kaggle control lane before returning to the paper-first remote-sensing dataset track.
- Hypothesis for `vR.1`: a conservative SR3 baseline should beat bicubic qualitatively if the script path, checkpointing, and metric logging are wired correctly.


## Dataset and Config Contract

- Full training config: `configs/phase1_sr3_vr1_kaggle.yaml`
- Smoke training config: `configs/phase1_sr3_vr1_smoke.yaml`
- Bicubic control config: `configs/phase0_bicubic_vr1_kaggle_control.yaml`
- Dataset contract: explicit paired LR/HR loader, fixed `4x` scale, repo-relative artifact paths
- Run-name contract: `vR.1-control`, `vR.1-smoke`, `vR.1-smoke-eval`, `vR.1-train`, `vR.1-eval`


## Table of Contents

1. Runtime diagnostics
2. W&B setup
3. Notebook readiness validation
4. Bicubic control evaluation
5. Smoke train plus smoke eval
6. Full train plus full eval
7. Results dashboard
8. Troubleshooting and handoff


## Weights & Biases Setup

Offline tracking is the safe default. If Kaggle secrets include a W&B key, you can still authenticate here before switching the config to online mode.


In [ ]:
import os
import subprocess

VERSION = 'vR.1'
NOTEBOOK_STEM = 'vR.1_HNDSR'
CONFIGS = {
    'control': 'configs/phase0_bicubic_vr1_kaggle_control.yaml',
    'smoke': 'configs/phase1_sr3_vr1_smoke.yaml',
    'train': 'configs/phase1_sr3_vr1_kaggle.yaml',
}
RUN_NAMES = {
    'control': 'vR.1-control',
    'smoke': 'vR.1-smoke',
    'smoke_eval': 'vR.1-smoke-eval',
    'train': 'vR.1-train',
    'eval': 'vR.1-eval',
}
ARTIFACT_ROOT = REPO_ROOT / 'artifacts'


def run_command(args):
    print(' '.join(str(arg) for arg in args))
    completed = subprocess.run(args, cwd=REPO_ROOT, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')


def _get_secret(name, aliases=None):
    aliases = aliases or [name]
    for candidate in aliases:
        value = os.environ.get(candidate)
        if value:
            return value
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(candidate)
        except Exception:
            continue
    return None


wandb_key = _get_secret('WANDB_API_KEY', aliases=['WANDB_API_KEY', 'wandb_api_key'])
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    print('W&B key detected. Tracking stays offline unless you change the config to online.')
else:
    print('No W&B key detected. Offline tracker mode remains the safe default.')


## Training Execution

Run the readiness validator before touching training. Do not skip the control lane or the smoke lane on the first Kaggle pass.


In [ ]:
run_command([
    PYTHON,
    'scripts/validate_notebook_version.py',
    '--version', VERSION,
    '--notebook', 'notebooks/versions/vR.1_HNDSR.ipynb',
    '--doc', 'docs/notebooks/vR.1_HNDSR.md',
    '--review', 'reports/reviews/vR.1_HNDSR.review.md',
    '--config', CONFIGS['train'],
    '--smoke-config', CONFIGS['smoke'],
    '--control-config', CONFIGS['control'],
])


In [ ]:
run_command([
    PYTHON,
    'scripts/evaluate_run.py',
    '--config', CONFIGS['control'],
    '--run-name', RUN_NAMES['control'],
])


In [ ]:
run_command([
    PYTHON,
    'scripts/train_baseline.py',
    '--config', CONFIGS['smoke'],
    '--run-name', RUN_NAMES['smoke'],
])

SMOKE_CHECKPOINT = ARTIFACT_ROOT / RUN_NAMES['smoke'] / 'checkpoints' / 'vR.1_sr3_smoke.pt'
print(f'Smoke checkpoint: {SMOKE_CHECKPOINT}')


In [ ]:
run_command([
    PYTHON,
    'scripts/evaluate_run.py',
    '--config', CONFIGS['smoke'],
    '--run-name', RUN_NAMES['smoke_eval'],
    '--checkpoint', str(SMOKE_CHECKPOINT),
])


In [ ]:
run_command([
    PYTHON,
    'scripts/train_baseline.py',
    '--config', CONFIGS['train'],
    '--run-name', RUN_NAMES['train'],
])

FULL_CHECKPOINT = ARTIFACT_ROOT / RUN_NAMES['train'] / 'checkpoints' / 'vR.1_sr3_best.pt'
print(f'Full checkpoint: {FULL_CHECKPOINT}')


## Evaluation and Export

This stage should export the same fixed smoke-pack comparisons and JSON summaries that the review pass will inspect later.


In [ ]:
run_command([
    PYTHON,
    'scripts/evaluate_run.py',
    '--config', CONFIGS['train'],
    '--run-name', RUN_NAMES['eval'],
    '--checkpoint', str(FULL_CHECKPOINT),
])


## Results Dashboard

The next cell reads local JSON summaries and displays the exported comparison grid if it exists.


In [ ]:
import json

from IPython.display import Image, display


def read_summary(run_name: str, filename: str):
    path = ARTIFACT_ROOT / run_name / 'metrics' / filename
    if not path.exists():
        print(f'Missing summary: {path}')
        return None
    payload = json.loads(path.read_text(encoding='utf-8'))
    print(f'===== {run_name} =====')
    print(json.dumps(payload, indent=2))
    return payload


read_summary(RUN_NAMES['control'], 'eval_summary.json')
read_summary(RUN_NAMES['smoke'], 'train_summary.json')
read_summary(RUN_NAMES['smoke_eval'], 'eval_summary.json')
read_summary(RUN_NAMES['train'], 'train_summary.json')
read_summary(RUN_NAMES['eval'], 'eval_summary.json')

grid_path = ARTIFACT_ROOT / RUN_NAMES['eval'] / 'samples' / 'vR.1_sr3_grid.png'
if grid_path.exists():
    display(Image(filename=str(grid_path)))
else:
    print(f'Grid not found yet: {grid_path}')


## Troubleshooting and Known Failure Modes

- If `validate_notebook_version.py` fails, stop and fix the notebook or paired docs before training.
- If Kaggle cannot see the repo paths, inspect `REPO_ROOT` in the first setup cell and adjust the working directory before rerunning.
- If W&B is not authenticated, keep offline mode and do not block the run on logging.
- If the smoke checkpoint does not appear, inspect the printed command output before moving to the full run.


## Changelog

- `vR.1`: initial immutable Kaggle notebook scaffold for the scratch SR3 baseline.
- Notebook-only logic is intentionally limited to diagnostics, subprocess orchestration, and results display.


## Next Step Gate

Do not fork `vR.2` until the executed `vR.1` notebook, its paired metrics, and its review doc all agree on one of these outcomes:

- freeze `vR.1` and promote to the next version
- patch `vR.1` in place for runtime-only fixes
- preserve a failure snapshot and document why the run is not promotable yet
